In [2]:
# ChromaDB 하려면 설치하고 써야 함
# %pip install chromadb llama_index.vector_stores.chroma llama-index-llms-ollama llama-index-embeddings-ollama

In [3]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [4]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model = 'gemma2:2b',
    temperature = 0.5, # 생성되는 텍스트의 다양성을 조절하는 매개변수
    request_timeout = 120.0 # 요청이 타임아웃되기까지의 시간(초)_이 시간이 되면 멈춰라
)

embed_model = OllamaEmbedding(
    model_name = 'nomic-embed-text'
)

In [6]:
# 데이터 로드
documents = SimpleDirectoryReader('../Data/pdf_sample2').load_data()

In [7]:
# 벡터 DB 생성 및 저장
db = chromadb.PersistentClient(path="./chroma_db") 

# 컬렉션 생성 및 저장
chroma_collection = db.get_or_create_collection("quickstart_ollama") # 스키마 정의

In [8]:
# ChromaDB를 LlamaIndex의 인덱싱 및 검색 파이프라인에 통합

vector_store = ChromaVectorStore(chroma_collection=chroma_collection) # ChromaVectorStore 생성
storage_context = StorageContext.from_defaults(vector_store=vector_store) # 스토리지 컨텍스트 생성
# 연결할 준비를 한 것

In [9]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(
    documents, # 데이터 
    storage_context = storage_context, # 스토리지 컨텍스트
    embed_model = embed_model,
    show_progress = True # 진행 상황 표시 여부
    )

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 12/12 [00:02<00:00,  4.33it/s]


---
### 메모리 로드

In [10]:
# 쿼리 엔진
query_engine = index.as_query_engine(llm = llm)

2026-05-19 11:51:02,653 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [11]:
# 쿼리 실행
query = "이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요."
response = query_engine.query(query)
# 응답 출력
print("\n질문: ", query)
print("답변: ", response)
# 더 자세한 답변을 원하면 파라메터 중 temperature 값을 높이면 된다 

2026-05-19 11:51:11,627 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:51:21,224 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문:  이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요.
답변:  이 논문에서는 Transformer 모델이 기존 모델보다 더 좋은 성능을 보였으며, 특히 English-to-German 및 English-to-French 번역에서 높은 BLEU 점수를 달성했습니다. 또한, 이 모델은 다른 모델과 비교했을 때 훨씬 빠르게 학습되었습니다.  

